# Process DB15K Data with Precomputed Embeddings

This notebook processes DB15K data using **precomputed embeddings** (no scraping required).

## Data Structure

```
raw/
  ├── train, valid, test           # Triple files (TSV format)
  ├── DB15k-entities2id.pkl        # Mapping: entity URI -> embedding index
  └── embeddings/
      ├── DB15K-textual.pth        # Precomputed entity text embeddings
      └── DB15K-visual.pth         # Precomputed entity visual embeddings
```

## Architecture Note

This notebook generates:
- Entity text embeddings (from precomputed)
- Entity visual embeddings (from precomputed)
- Entity image mask
- Triple tensors with entity/relation IDs
- Vocabularies (entity2id, relation2id)

## Requirements

```bash
pip install torch tqdm
```

## 1. Configuration

In [ ]:
import json
import pickle
import torch
import torch.nn.functional as F
from pathlib import Path
from tqdm.auto import tqdm

# ============================================================================
# PATHS
# ============================================================================

RAW_DIR = Path(r"D:\NLP research\Code\graph-world-models\GWM\gwm-rnn\data\link-prediction\multimodal\db15k\raw")
OUTPUT_DIR = Path(r"D:\NLP research\Code\graph-world-models\GWM\gwm-rnn\data\link-prediction\multimodal\db15k\processed")

# Create output directories
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / 'triples').mkdir(exist_ok=True)
(OUTPUT_DIR / 'embeddings').mkdir(exist_ok=True)

# ============================================================================
# DATASET INFO
# ============================================================================

DATASET_NAME = 'DB15K'
NORMALIZE_EMBEDDINGS = True  # Normalize to unit sphere
CREATE_INVERSE_RELATIONS = True  # Add inverse triples (h, r, t) -> (t, r_inv, h)

print("="*70)
print(f"PROCESSING {DATASET_NAME} DATA")
print("="*70)
print(f"Raw data: {RAW_DIR}")
print(f"Output: {OUTPUT_DIR}")
print(f"Normalize embeddings: {NORMALIZE_EMBEDDINGS}")
print(f"Create inverse relations: {CREATE_INVERSE_RELATIONS}")
print("="*70)

PROCESSING DB15K DATA
Raw data: D:\NLP research\Code\graph-world-models\GWM\gwm-rnn\data\link-prediction\multimodal\db15k\raw
Output: D:\NLP research\Code\graph-world-models\GWM\gwm-rnn\data\link-prediction\multimodal\db15k\processed
Normalize embeddings: True


## 2. Load and Parse Triple Files

In [ ]:
print("\n[1/5] Loading triple files...")

def parse_triple_file(file_path):
    """Parse TSV triple file and extract unique entities and relations."""
    triples = []
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            parts = line.split('\t')
            if len(parts) != 3:
                continue
            h, r, t = parts
            triples.append((h, r, t))
    return triples

# Load all splits
train_triples_raw = parse_triple_file(RAW_DIR / 'train')
valid_triples_raw = parse_triple_file(RAW_DIR / 'valid')
test_triples_raw = parse_triple_file(RAW_DIR / 'test')

all_triples_raw = train_triples_raw + valid_triples_raw + test_triples_raw

print(f"✓ Loaded triples:")
print(f"  Train: {len(train_triples_raw):,}")
print(f"  Valid: {len(valid_triples_raw):,}")
print(f"  Test: {len(test_triples_raw):,}")
print(f"  Total: {len(all_triples_raw):,}")

## 3. Load Entity Mapping and Build Vocabularies

In [ ]:
print("\n[2/5] Loading entity mapping...")

# Load precomputed entity mapping (entity URI -> embedding index)
entity_mapping_path = RAW_DIR / 'DB15k-entities2id.pkl'
print(f"  Loading: {entity_mapping_path.name}")
with open(entity_mapping_path, 'rb') as f:
    uri_to_emb_idx = pickle.load(f)

print(f"  ✓ Loaded mapping for {len(uri_to_emb_idx):,} entities")

print("\n[3/5] Building vocabularies...")

# Extract unique entities and relations
entities = set()
relations = set()

for h, r, t in all_triples_raw:
    entities.add(h)
    entities.add(t)
    relations.add(r)

# Add inverse relations if enabled
if CREATE_INVERSE_RELATIONS:
    original_relations = list(relations)
    for rel in original_relations:
        relations.add(rel + '_inv')
    print(f"  Added {len(original_relations)} inverse relations")

# Create sorted mappings
entities_sorted = sorted(entities)
relations_sorted = sorted(relations)

entity2id = {e: i for i, e in enumerate(entities_sorted)}
relation2id = {r: i for i, r in enumerate(relations_sorted)}

print(f"✓ Created vocabularies:")
print(f"  Entities: {len(entity2id):,}")
print(f"  Relations: {len(relation2id):,} (original: {len(original_relations) if CREATE_INVERSE_RELATIONS else len(relation2id)})")

# Convert triples to ID format with optional inverse triples
def convert_triples(triples_raw, add_inverse=False):
    triples = []
    for h, r, t in triples_raw:
        h_id = entity2id[h]
        r_id = relation2id[r]
        t_id = entity2id[t]
        triples.append((h_id, r_id, t_id))
        
        # Add inverse triple (only for training!)
        if add_inverse:
            r_inv_id = relation2id[r + '_inv']
            triples.append((t_id, r_inv_id, h_id))
    
    return triples

# Only add inverse relations to training split (standard practice)
train_triples = convert_triples(train_triples_raw, add_inverse=CREATE_INVERSE_RELATIONS)
valid_triples = convert_triples(valid_triples_raw, add_inverse=False)
test_triples = convert_triples(test_triples_raw, add_inverse=False)

train_triples_tensor = torch.tensor(train_triples, dtype=torch.long)
valid_triples_tensor = torch.tensor(valid_triples, dtype=torch.long)
test_triples_tensor = torch.tensor(test_triples, dtype=torch.long)

print(f"✓ Converted to tensor format:")
print(f"  Train: {train_triples_tensor.shape}")
print(f"  Valid: {valid_triples_tensor.shape}")
print(f"  Test: {test_triples_tensor.shape}")

## 4. Load and Reorder Precomputed Embeddings

In [ ]:
print("\n[4/5] Loading precomputed embeddings...")

# Load text embeddings
text_emb_path = RAW_DIR / 'embeddings' / 'DB15K-textual.pth'
print(f"  Loading: {text_emb_path.name}")
entity_text_embeddings = torch.load(text_emb_path, map_location='cpu')

# Handle different data formats
if isinstance(entity_text_embeddings, dict):
    if 'embeddings' in entity_text_embeddings:
        entity_text_embeddings = entity_text_embeddings['embeddings']
    elif 'ent_embeddings' in entity_text_embeddings:
        entity_text_embeddings = entity_text_embeddings['ent_embeddings']

# Convert to tensor if needed
if not isinstance(entity_text_embeddings, torch.Tensor):
    entity_text_embeddings = torch.tensor(entity_text_embeddings, dtype=torch.float32)

print(f"    Shape: {entity_text_embeddings.shape}")
print(f"    Dtype: {entity_text_embeddings.dtype}")

# Load visual embeddings
visual_emb_path = RAW_DIR / 'embeddings' / 'DB15K-visual.pth'
print(f"  Loading: {visual_emb_path.name}")
entity_visual_embeddings = torch.load(visual_emb_path, map_location='cpu')

# Handle different data formats
if isinstance(entity_visual_embeddings, dict):
    if 'embeddings' in entity_visual_embeddings:
        entity_visual_embeddings = entity_visual_embeddings['embeddings']
    elif 'ent_embeddings' in entity_visual_embeddings:
        entity_visual_embeddings = entity_visual_embeddings['ent_embeddings']

# Convert to tensor if needed
if not isinstance(entity_visual_embeddings, torch.Tensor):
    entity_visual_embeddings = torch.tensor(entity_visual_embeddings, dtype=torch.float32)

print(f"    Shape: {entity_visual_embeddings.shape}")
print(f"    Dtype: {entity_visual_embeddings.dtype}")

print(f"\n✓ Raw embeddings loaded: {entity_text_embeddings.shape}")

# Reorder embeddings to match our entity2id order
print("\n  Reordering embeddings to match entity2id...")
num_entities = len(entity2id)
text_dim = entity_text_embeddings.shape[1]
visual_dim = entity_visual_embeddings.shape[1]

# Create reordered tensors
reordered_text_embs = torch.zeros(num_entities, text_dim)
reordered_visual_embs = torch.zeros(num_entities, visual_dim)

# Map each entity to its embedding
for entity_uri, our_id in entity2id.items():
    if entity_uri in uri_to_emb_idx:
        emb_idx = uri_to_emb_idx[entity_uri]
        if emb_idx < entity_text_embeddings.shape[0]:
            reordered_text_embs[our_id] = entity_text_embeddings[emb_idx]
            reordered_visual_embs[our_id] = entity_visual_embeddings[emb_idx]
    # If entity not in mapping, embedding remains zeros

# Replace with reordered embeddings
entity_text_embeddings = reordered_text_embs
entity_visual_embeddings = reordered_visual_embs

print(f"  ✓ Reordered embeddings: {entity_text_embeddings.shape}")

## 5. Normalize and Create Modality Masks

In [ ]:
print("\n[5/5] Processing embeddings...")

# Normalize embeddings to unit sphere
if NORMALIZE_EMBEDDINGS:
    print("  Creating text mask...")
    # Create text mask BEFORE normalization (zero rows = missing text)
    entity_text_mask = (entity_text_embeddings.abs().sum(dim=1) > 1e-6)
    
    print("  Normalizing text embeddings...")
    # Only normalize non-zero rows
    if entity_text_mask.sum() > 0:
        entity_text_embeddings[entity_text_mask] = F.normalize(
            entity_text_embeddings[entity_text_mask],
            p=2,
            dim=1
        )
    
    print("  Creating image mask...")
    # Create image mask BEFORE normalization (zero rows = no image)
    entity_image_mask = (entity_visual_embeddings.abs().sum(dim=1) > 1e-6)
    
    print("  Normalizing visual embeddings...")
    # Only normalize non-zero rows
    if entity_image_mask.sum() > 0:
        entity_visual_embeddings[entity_image_mask] = F.normalize(
            entity_visual_embeddings[entity_image_mask],
            p=2,
            dim=1
        )
    
    # Verify normalization
    if entity_text_mask.sum() > 0:
        text_norms = torch.norm(entity_text_embeddings[entity_text_mask], p=2, dim=1)
        print(f"    Text norms: mean={text_norms.mean():.4f}, std={text_norms.std():.4f}")
    
    if entity_image_mask.sum() > 0:
        visual_norms = torch.norm(entity_visual_embeddings[entity_image_mask], p=2, dim=1)
        print(f"    Visual norms: mean={visual_norms.mean():.4f}, std={visual_norms.std():.4f}")
else:
    # Create masks without normalization
    entity_text_mask = (entity_text_embeddings.abs().sum(dim=1) > 1e-6)
    entity_image_mask = (entity_visual_embeddings.abs().sum(dim=1) > 1e-6)

print(f"\n✓ Modality coverage:")
print(f"  Text embeddings:")
print(f"    WITH text: {entity_text_mask.sum().item():,} ({entity_text_mask.float().mean()*100:.1f}%)")
print(f"    WITHOUT text (will use <MISSING_TEXT> token): {(~entity_text_mask).sum().item():,} ({(~entity_text_mask).float().mean()*100:.1f}%)")
print(f"  Image embeddings:")
print(f"    WITH images: {entity_image_mask.sum().item():,} ({entity_image_mask.float().mean()*100:.1f}%)")
print(f"    WITHOUT images (will use <MISSING_IMG> token): {(~entity_image_mask).sum().item():,} ({(~entity_image_mask).float().mean()*100:.1f}%)")

## 6. Save Processed Data

In [ ]:
print("\n[6/6] Saving processed data...")

# Save triples
torch.save(train_triples_tensor, OUTPUT_DIR / 'triples' / 'train.pt')
torch.save(valid_triples_tensor, OUTPUT_DIR / 'triples' / 'valid.pt')
torch.save(test_triples_tensor, OUTPUT_DIR / 'triples' / 'test.pt')
print("  ✓ Saved triples")

# Save entity embeddings and masks
torch.save(entity_text_embeddings, OUTPUT_DIR / 'embeddings' / 'entity_text.pt')
torch.save(entity_visual_embeddings, OUTPUT_DIR / 'embeddings' / 'entity_image.pt')
torch.save(entity_text_mask, OUTPUT_DIR / 'embeddings' / 'entity_text_mask.pt')
torch.save(entity_image_mask, OUTPUT_DIR / 'embeddings' / 'entity_image_mask.pt')
print("  ✓ Saved embeddings and masks")
print(f"    entity_text.pt: {entity_text_embeddings.shape}")
print(f"    entity_image.pt: {entity_visual_embeddings.shape}")
print(f"    entity_text_mask.pt: {entity_text_mask.shape}")
print(f"    entity_image_mask.pt: {entity_image_mask.shape}")

# Save vocabularies
with open(OUTPUT_DIR / 'entity2id.json', 'w', encoding='utf-8') as f:
    json.dump(entity2id, f, indent=2, ensure_ascii=False)

with open(OUTPUT_DIR / 'relation2id.json', 'w', encoding='utf-8') as f:
    json.dump(relation2id, f, indent=2, ensure_ascii=False)
print("  ✓ Saved vocabularies")

# Save metadata
metadata = {
    'dataset_name': DATASET_NAME,
    'source': 'Precomputed embeddings from previous work',
    'num_entities': len(entity2id),
    'num_relations': len(relation2id),
    'num_train_triples': len(train_triples),
    'num_valid_triples': len(valid_triples),
    'num_test_triples': len(test_triples),
    'text_embedding_dim': int(entity_text_embeddings.shape[1]),
    'visual_embedding_dim': int(entity_visual_embeddings.shape[1]),
    'text_coverage': float(entity_text_mask.float().mean()),
    'image_coverage': float(entity_image_mask.float().mean()),
    'normalized': NORMALIZE_EMBEDDINGS,
    'inverse_relations': CREATE_INVERSE_RELATIONS
}

with open(OUTPUT_DIR / 'metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)
print("  ✓ Saved metadata")

print("\n" + "="*70)
print("✅ DATA PROCESSING COMPLETE!")
print("="*70)
print(f"\nOutput: {OUTPUT_DIR}")
print("\nGenerated files:")
print("  triples/")
print("    train.pt, valid.pt, test.pt")
print("  embeddings/")
print("    entity_text.pt, entity_image.pt")
print("    entity_text_mask.pt, entity_image_mask.pt")
print("  entity2id.json, relation2id.json")
print("  metadata.json")

## 7. Verification

In [ ]:
print("="*70)
print("VERIFICATION")
print("="*70)

def check_file(path, description):
    if path.exists():
        size_mb = path.stat().st_size / (1024**2)
        print(f"✓ {description:<40} ({size_mb:>6.2f} MB)")
        return True
    else:
        print(f"✗ {description:<40} MISSING")
        return False

print("\nChecking files:")
all_ok = True
all_ok &= check_file(OUTPUT_DIR / 'triples' / 'train.pt', 'triples/train.pt')
all_ok &= check_file(OUTPUT_DIR / 'triples' / 'valid.pt', 'triples/valid.pt')
all_ok &= check_file(OUTPUT_DIR / 'triples' / 'test.pt', 'triples/test.pt')
all_ok &= check_file(OUTPUT_DIR / 'embeddings' / 'entity_text.pt', 'entity_text.pt')
all_ok &= check_file(OUTPUT_DIR / 'embeddings' / 'entity_image.pt', 'entity_image.pt')
all_ok &= check_file(OUTPUT_DIR / 'embeddings' / 'entity_text_mask.pt', 'entity_text_mask.pt')
all_ok &= check_file(OUTPUT_DIR / 'embeddings' / 'entity_image_mask.pt', 'entity_image_mask.pt')
all_ok &= check_file(OUTPUT_DIR / 'entity2id.json', 'entity2id.json')
all_ok &= check_file(OUTPUT_DIR / 'relation2id.json', 'relation2id.json')
all_ok &= check_file(OUTPUT_DIR / 'metadata.json', 'metadata.json')

if all_ok:
    print("\n✅ All files present!")
else:
    print("\n⚠️  Some files missing")

print("\n" + "="*70)
print("STATISTICS")
print("="*70)
print(f"\nEntities: {len(entity2id):,}")
print(f"Relations: {len(relation2id):,}")
print(f"\nTriples:")
print(f"  Train: {len(train_triples):,}")
print(f"  Valid: {len(valid_triples):,}")
print(f"  Test: {len(test_triples):,}")
print(f"\nEmbeddings:")
print(f"  Text dimension: {entity_text_embeddings.shape[1]}")
print(f"  Visual dimension: {entity_visual_embeddings.shape[1]}")
print(f"  Text coverage: {entity_text_mask.float().mean()*100:.1f}%")
print(f"  Image coverage: {entity_image_mask.float().mean()*100:.1f}%")
print(f"  Normalized: {NORMALIZE_EMBEDDINGS}")
print(f"\n✅ Ready for context generation and training!")

## 8. Generate context embeddings

In [ ]:
import os

required_files = ['generate_context_embeddings.py']

print("="*70)
print("Cloning GitHub repository...")
print("="*70)

# Clone your GitHub repo
GITHUB_REPO = "https://github.com/HiIamPhuc/GWM.git"
BRANCH = "no-rel-text"

!git clone {GITHUB_REPO} /kaggle/working/gwm
%cd /kaggle/working/gwm
!git checkout {BRANCH}
!git pull
%cd ../

# Copy files from repo to working directory
repo_path = "/kaggle/working/gwm/gwm-rnn/link-prediction/multimodal"

print(f"\nCopying files from {repo_path}...")
for file in required_files:
    !cp {repo_path}/{file} /kaggle/working/
    print(f"✓ Copied {file}")

# Verify files exist
base_path = '/kaggle/working'

missing_files = []
for file in required_files:
    file_path = os.path.join(base_path, file)
    if not os.path.exists(file_path):
        missing_files.append(file)

if missing_files:
    print(f"\n⚠️  Some files not found: {missing_files}")
    print("Training scripts will be imported from parent directory")
else:
    print(f"\n✓ All required files ready: {required_files}")

In [ ]:
!python generate_context_embeddings.py \
    --data_dir processed/ \
    --aggregation mean \
    --top_k 20